In [3]:
# import the required packages
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
import os
from pprint import pprint

In [4]:
# load the environment variables
load_dotenv()

hf_token = os.getenv('HF_TOKEN')

In [5]:
# Sentence-Transformers supports many models - the right one depends on the task (QA), language (English) and resources (less memory) one has
model = SentenceTransformer(model_name_or_path='all-MiniLM-L6-v2', token=hf_token) # 384-dim vectors - fast on CPU - good quality for general English text - uses cosine similarity

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
# create embeddings of a sample user query
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)
print(f'Shape of question1 vector: {v1.shape}')
# create embeddings of a document similar to the user query
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)
print(f'Shape of document vector: {dv.shape}') # same dimensions as the query (384,)
# calculate dot product between query and document
print(f'Cosine similarity: {v1.dot(dv):.2f}')
# create embeddings of an unrelated sample user query
q2 = "Who will win grand prix 2026"
v2 = model.encode(q2)
print(f'Shape of question2 vector: {v2.shape}')
# calculate dot product between query and document
print(f'Cosine similarity: {v2.dot(dv):.2f}') # grand prix 2026 has nothing to do with the course registration

Shape of question1 vector: (384,)
Shape of document vector: (384,)
Cosine similarity: 0.32
Shape of question2 vector: (384,)
Cosine similarity: -0.01


In [7]:
# donwload the ingest script from Week01-RAG
# !wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

In [8]:
import numpy as np
from ingest import load_faq_data
from tqdm.auto import tqdm

documents = load_faq_data()
pprint(documents[0])

{'answer': 'A new cohort runs roughly January–April every year. For the '
           "current cohort's exact start date and registration link, check the "
           '[course repo '
           'README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n'
           '\n'
           '- Register via the link in the course repo before the cohort '
           'starts.\n'
           '- Join the [course Telegram channel](https://t.me/dezoomcamp) for '
           'announcements.\n'
           "- Join DataTalks.Club's "
           '[Slack](https://datatalks.club/docs/general/slack/) and the '
           '`#course-data-engineering` channel.',
 'course': 'data-engineering-zoomcamp',
 'doc_id': '9e508f2212',
 'question': 'Course: When does the course start?',
 'section': 'General Course-Related Questions'}


In [9]:
# embed question and answer together - that way a querry can match the question text and answer text in our index
texts = [] # corpus of 'question answer' from documents
for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [10]:
# handing the entire text to the embedding model takes a long time and we can't see the progress; instead split them into batches
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

print(f'Number of documents: {len(texts)}')
print(f'Number of document embeddings: {len(vectors)}')

  0%|          | 0/27 [00:00<?, ?it/s]

Number of documents: 1350
Number of document embeddings: 1350


In [11]:
# convert the list of document embeddings to a numpy array of size (1350, 384)
X = np.array(vectors)
print(f'Embedding Matrix size: {X.shape}')

Embedding Matrix size: (1350, 384)


In [12]:
# perform matrix-vector multiplication between document embedding matrix and query vector to calculate the similarity scores and return the most similar documents 
# (in other words, to perform the vector search); we could use for loop to go over each document embedding, perform vector-vector dot product between the document vector and query vector,
# and record the scores; however, the matrix-vector multiplication is vectorized and numpy runs an optimized C code instead of looping in python
query = 'Can I still join the course after the start date?'
v_query = model.encode(query)

scores = X.dot(v_query)
# argmax returns the row which has the maximum score
idx = np.argmax(scores)
print(f'Index: {idx}\nScore: {scores[idx]}')
pprint(f'Document: {documents[idx]}')

Index: 2
Score: 0.7629410028457642
("Document: {'course': 'data-engineering-zoomcamp', 'section': 'General "
 "Course-Related Questions', 'question': 'Course: Can I still join the course "
 'after the start date?\', \'answer\': "Yes, even if you don\'t register, '
 "you're still eligible to submit the homework.\\n\\nBe aware, however, that "
 'there will be deadlines for turning in homeworks and the final projects. So '
 'don\'t leave everything for the last minute.", \'doc_id\': \'3f1424af17\'}')


In [13]:
# usually we want more than a single match as the answer to a question can be spread across several documents; sometimes the top result isn't the right onebut second is
# hence, we send all the top most similar doucmnets and let LLM combine them
top5 = np.argsort(-scores)[:5] # argsort returns the indices of values sorted in ascending order; hence, -scores will output the indices in decreasing order of values - we select first 5 of these
# here are the actual documents
for idx in top5:
    print(scores[idx])
    pprint(documents[idx])
    print()

0.762941
{'answer': "Yes, even if you don't register, you're still eligible to submit "
           'the homework.\n'
           '\n'
           'Be aware, however, that there will be deadlines for turning in '
           "homeworks and the final projects. So don't leave everything for "
           'the last minute.',
 'course': 'data-engineering-zoomcamp',
 'doc_id': '3f1424af17',
 'question': 'Course: Can I still join the course after the start date?',
 'section': 'General Course-Related Questions'}

0.7579371
{'answer': "Yes, even if you don't register, you're still eligible to submit "
           'the homeworks as long as the form is still open and accepting '
           'submissions.\n'
           '\n'
           'Be aware, however, that there will be deadlines for turning in the '
           "final projects. So don't leave everything to the last minute.",
 'course': 'mlops-zoomcamp',
 'doc_id': '2d8b16c2a0',
 'question': 'Course - Can I still join the course after the start date?'

In [14]:
# it might happen that we want to limit our document search using a particular filter (example, course = 'llm-zoomcamp'); instead of manually filtering it and performing similarity search,
# use optimized vector db - here we use minsearch, which is a wrapper to what we did manually with numpy - other dbs such as chroma, qdrant will be more optimized
from minsearch import VectorSearch

In [15]:
# index the document vectors
vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(vectors=X, payload=documents) # payload are the actual documents or metadata associated with each embedding; search engine keeps this mapping

In [ ]:
# perform document-query similarity search using minsearch index
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vindex.search(
    query_vector=query_vector, 
    filter_dict={'course':'llm-zoomcamp'},
    num_results=5
)